World Happiness Report 2023

In [20]:
import pandas as pd
import numpy as np
from google.colab import files # Import files module

print("Please upload the 'world_happiness_2023.csv' file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

original_df = pd.read_csv(file_name)
original_df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(original_df)} countries, {len(original_df.columns)} columns")
print(original_df.head())

Please upload the 'world_happiness_2023.csv' file:


Saving world_happiness_2023.csv to world_happiness_2023 (4).csv
Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [16]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(original_df['Region'].value_counts()) # Use original_df
print("\nScore range:", original_df['Happiness_Score'].min(), "–", original_df['Happiness_Score'].max()) # Use original_df
print("\nBottom 10 countries:")
print(original_df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']]) # Use original_df

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

Task 1: Regional comparison bar chart

In [17]:
import plotly.express as px
import pandas as pd

# The dummy DataFrame creation below was removed to ensure that the 'original_df'
# loaded from 'world_happiness_2023.csv' in cell JlOcWcbWuALJ is used.
# data = {
#     'Region': ['Western Europe', 'North America', 'Australia and New Zealand', 'Latin America and Caribbean', 'Eastern Asia', 'Central and Eastern Europe', 'Southeast Asia', 'Middle East and Northern Africa', 'Sub-Saharan Africa', 'Southern Asia'],
#     'Happiness_Score': [7.5, 7.3, 7.3, 6.1, 5.7, 5.5, 5.3, 5.2, 4.3, 3.8]
# }
# df = pd.DataFrame(data)

#Compute average happiness score by region
region_avg = (
    original_df.groupby('Region')['Happiness_Score'] # Use original_df
      .mean()
      .reset_index()
      .sort_values('Happiness_Score', ascending=True)   # ascending for horizontal bar order
)

print(region_avg)

#Build the chart

fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text='Happiness_Score',
    color='Happiness_Score',   # adds colour variation beyond default
    color_continuous_scale='Viridis'
)

# Improve layout and design
fig.update_layout(
    title={
        'text': 'Western Europe Stands Out as the Happiest Region — Strong Social Support and Economic Stability Matter',
        'x': 0.5
    },
    xaxis_title='Average Happiness Score',
    yaxis_title='Region',
    xaxis=dict(range=[0, region_avg['Happiness_Score'].max() + 1]),  # zero baseline
    template='plotly_white',
    height=600
)

# Format labels
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

# Add annotation for top region
top_region = region_avg.iloc[-1]

fig.add_annotation(
    x=top_region['Happiness_Score'],
    y=top_region['Region'],
    text='Highest average happiness',
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=0
)

# Show chart
fig.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


Task 2: Top 8 vs. Bottom 8 contrast

In [19]:
import pandas as pd
import plotly.express as px

# Step 1: Get top and bottom countries
top8 = original_df.nlargest(8, 'Happiness_Score').copy() # Use original_df
top8['Group'] = 'Top 8'

bottom8 = original_df.nsmallest(8, 'Happiness_Score').copy() # Use original_df
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')

# Global average
global_avg = original_df['Happiness_Score'].mean() # Use original_df
print(f"Global average: {global_avg:.2f}")

# Step 2: Build the chart

fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Region', # Changed 'Country' to 'Region'
    orientation='h',
    color='Group',
    text='Happiness_Score',
    color_discrete_map={
        'Top 8': '#2E8B57',      # green
        'Bottom 8': '#C0392B'    # red
    }
)

# Improve labels
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

# Add global average reference line
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='black',
    annotation_text=f'Global Avg = {global_avg:.2f}',
    annotation_position='top'
)

# Add annotation to emphasise the gap
fig.add_annotation(
    x=global_avg,
    y=7.5,
    text='Large happiness gap between countries',
    showarrow=True,
    arrowhead=2,
    ax=80,
    ay=-40
)

# Layout improvements
fig.update_layout(
    title={
        'text': 'Nordic Countries Lead While Fragile States Lag Far Behind — The Happiness Gap Reflects Inequality in Stability and Quality of Life',
        'x': 0.5
    },
    xaxis_title='Happiness Score',
    yaxis_title='Region', # Changed 'Country' to 'Region'
    template='plotly_white',
    height=700,
    legend_title='Group'
)

# Ensure x-axis starts at zero
fig.update_xaxes(range=[0, combined['Happiness_Score'].max() + 1])

# Show chart
fig.show()

Global average: 5.81
